In [4]:
def print_board(state):
    """Prints the 1D tuple state as a 3x3 grid."""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        # Replace 0 with an underscore for better visibility
        formatted_row = [str(x) if x != 0 else '_' for x in row]
        print(" ".join(formatted_row))

def get_board_input(prompt_name):
    """Takes user input to build the 3x3 board."""
    print(f"\nEnter {prompt_name} state:")
    print("(Type 3 numbers per line separated by spaces. Use 0 for the empty space)")
    board = []
    for i in range(3):
        while True:
            try:
                row = input(f"Row {i+1}: ").strip().split()
                if len(row) != 3:
                    print("Please enter exactly 3 numbers.")
                    continue
                board.extend([int(x) for x in row])
                break
            except ValueError:
                print("Invalid input. Please enter numbers only.")
    return tuple(board)

def get_neighbors(state):
    """Generates valid moves (Up, Down, Left, Right) from the current state."""
    neighbors = []
    zero_index = state.index(0)
    row, col = divmod(zero_index, 3)

    moves = {
        'Up': (row - 1, col),
        'Down': (row + 1, col),
        'Left': (row, col - 1),
        'Right': (row, col + 1)
    }

    for move_name, (r, c) in moves.items():
        if 0 <= r < 3 and 0 <= c < 3:
            new_index = r * 3 + c
            new_state = list(state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
            neighbors.append((tuple(new_state), move_name))
            
    return neighbors

def depth_limited_search(start_state, goal_state, limit):
    """Runs DFS but strictly stops exploring if it hits the depth limit."""
    # Stack stores: (current_state, list_of_moves, list_of_states_visited)
    stack = [(start_state, [], [start_state])]
    
    # Track the shortest depth at which we visited a state to optimize search
    visited = {start_state: 0}
    nodes_evaluated = 0

    while stack:
        current_state, moves, states = stack.pop()
        depth = len(moves)
        nodes_evaluated += 1

        if current_state == goal_state:
            return moves, states, nodes_evaluated

        if depth < limit:
            for next_state, move in reversed(get_neighbors(current_state)):
                # Only explore if it's new, or if we found a shorter path to it
                if next_state not in visited or visited[next_state] > depth + 1:
                    visited[next_state] = depth + 1
                    stack.append((next_state, moves + [move], states + [next_state]))

    return None, None, nodes_evaluated

def ids_8_puzzle(start_state, goal_state, max_depth=35):
    """Repeatedly runs depth-limited search, increasing the limit by 1 each time."""
    total_nodes_evaluated = 0
    
    for limit in range(max_depth + 1):
        print(f"Searching at depth limit {limit}...")
        moves, states, nodes = depth_limited_search(start_state, goal_state, limit)
        total_nodes_evaluated += nodes
        
        if moves is not None:
            return moves, states, total_nodes_evaluated

    return None, None, total_nodes_evaluated

# --- Execution ---
if __name__ == "__main__":
    start = get_board_input("START")
    goal = get_board_input("GOAL")

    print("\nRunning Iterative Deepening Search (IDS)...")
    moves, states, nodes_visited = ids_8_puzzle(start, goal)

    if moves is not None:
        print(f"\nGoal found in {len(moves)} moves! (Total nodes evaluated: {nodes_visited})\n")
        
        print("--- PATH RECONSTRUCTION ---")
        print("Start State:")
        print_board(states[0])
        
        for i, move in enumerate(moves):
            print(f"\nStep {i + 1}: Move empty space {move}")
            print_board(states[i + 1])
    else:
        print(f"\nNo solution found within the maximum depth limit. Nodes evaluated: {nodes_visited}")
        print("Note: Half of all random 8-puzzle configurations are mathematically unsolvable.")


Enter START state:
(Type 3 numbers per line separated by spaces. Use 0 for the empty space)


Row 1:  1 2 3
Row 2:  4 5 6
Row 3:  7 0 8



Enter GOAL state:
(Type 3 numbers per line separated by spaces. Use 0 for the empty space)


Row 1:  1 2 3
Row 2:  4 5 6
Row 3:   7 8 0



Running Iterative Deepening Search (IDS)...
Searching at depth limit 0...
Searching at depth limit 1...

Goal found in 1 moves! (Total nodes evaluated: 5)

--- PATH RECONSTRUCTION ---
Start State:
1 2 3
4 5 6
7 _ 8

Step 1: Move empty space Right
1 2 3
4 5 6
7 8 _
